In [1]:
import pathlib

import pandas as pd
import swifter

from rdkit import Chem
from rdkit.Chem import Descriptors
from tqdm.auto import tqdm

In [ ]:
NOTEBOOK_DIR = pathlib.Path().absolute()
DATA_DIR = NOTEBOOK_DIR.parents[1] / "data" / "solvation"
QUANTUM_GREEN_DIR = pathlib.Path("/home/shared/projects/quantum_green")
PAPER_DATA_DIR = QUANTUM_GREEN_DIR / "paper" / "data"

In [3]:
def molecule_from_smiles(smiles, remove_atom_mapping=False):
    mol = Chem.MolFromSmiles(smiles)
    if remove_atom_mapping:
        for atom in mol.GetAtoms():
            atom.SetAtomMapNum(0)
    return mol


def canonical_smiles_from_molecule(mol, isomeric=True):
    return Chem.MolToSmiles(mol, isomericSmiles=isomeric)


def first_letter(name):
    return next(s for s in name if s.isalpha())


pd.set_option("display.max_columns", None)


def head(df, n=2):
    display(df.head(n))
    print(f"Contains {len(df)} rows")


def swifter_apply(series, func, desc=None):
    if desc is None:
        desc = "Applying function"
    return series.swifter.progress_bar(True, desc).apply(func)


In [4]:
names_df = pd.read_csv(pathlib.Path.cwd() / "solvents.csv")
head(names_df)

,cosmo_name,smiles,inchi,cosmo_conf,source,exp_dielectric,T,source.1,polar,protic
0,"(1,1-dimethylethyl)benzene",CC(C)(C)c1ccccc1,"InChI=1/C10H14/c1-10(2,3)9-7-5-4-6-8-9/h4-8H,1...",1,COSMObase,2.359,20.0,NaN,0.0,0
1,"1-(1,1-dimethylethoxy)-2-propanol",CC(O)COC(C)(C)C,"InChI=1/C7H16O2/c1-6(8)5-9-7(2,3)4/h6,8H,5H2,1...",5,COSMObase,NaN,NaN,NaN,NaN,1


Contains 295 rows


In [5]:
non_isomeric_canonical_smiles = names_df["smiles"].apply(
    lambda x: canonical_smiles_from_molecule(molecule_from_smiles(x), isomeric=False)
)
(names_df["smiles"] != non_isomeric_canonical_smiles).sum()

np.int64(0)

In [6]:
smiles_to_name_mapping = names_df.set_index("smiles")["cosmo_name"].to_dict()

## Species Data

In [7]:
data = pd.read_csv(
    PAPER_DATA_DIR / "solvation" / "FILTERED_DEDUPLICATED_full_data_v3.csv",
    low_memory=False,
)
head(data)

,solvent_smiles,solute_name,solute_smiles,Gsolv (kcal/mol),Hsolv (kcal/mol),project,hash
0,O,id0,[O:1]([O:2][H:4])[H:3],-7.178090,-11.557472,aug11b,InChI=1/H2O2/c1-2/h1-2HInChI=1S/H2O/h1H2
1,CCCCCCCCO,id0,[O:1]([O:2][H:4])[H:3],-5.797229,-14.171550,aug11b,InChI=1/H2O2/c1-2/h1-2HInChI=1S/C8H18O/c1-2-3-...


Contains 101513860 rows


In [8]:
unique_solvents = pd.DataFrame(
    data["solvent_smiles"].unique(), columns=["solvent_smiles"]
)
unique_solvents["non_isomeric_canonical_smiles"] = unique_solvents[
    "solvent_smiles"
].apply(
    lambda x: canonical_smiles_from_molecule(molecule_from_smiles(x), isomeric=False)
)
head(unique_solvents)

,solvent_smiles,non_isomeric_canonical_smiles
0,O,O
1,CCCCCCCCO,CCCCCCCCO


Contains 295 rows


In [9]:
(
    unique_solvents["solvent_smiles"]
    != unique_solvents["non_isomeric_canonical_smiles"]
).sum()

np.int64(85)

In [10]:
solvent_smiles_to_non_isomeric_canonical_smiles_mapping = unique_solvents.set_index(
    "solvent_smiles"
)["non_isomeric_canonical_smiles"].to_dict()
solvent_smiles_to_name_mapping = {
    k: smiles_to_name_mapping[v]
    for k, v in solvent_smiles_to_non_isomeric_canonical_smiles_mapping.items()
}

In [11]:
data["solvent_name"] = data["solvent_smiles"].map(solvent_smiles_to_name_mapping)
head(data)

,solvent_smiles,solute_name,solute_smiles,Gsolv (kcal/mol),Hsolv (kcal/mol),project,hash,solvent_name
0,O,id0,[O:1]([O:2][H:4])[H:3],-7.178090,-11.557472,aug11b,InChI=1/H2O2/c1-2/h1-2HInChI=1S/H2O/h1H2,h2o
1,CCCCCCCCO,id0,[O:1]([O:2][H:4])[H:3],-5.797229,-14.171550,aug11b,InChI=1/H2O2/c1-2/h1-2HInChI=1S/C8H18O/c1-2-3-...,1-octanol


Contains 101513860 rows


In [12]:
unique_solutes = pd.DataFrame(
    data["solute_smiles"].unique(), columns=["atom_mapped_smiles"]
)
head(unique_solutes)

,atom_mapped_smiles
0,[O:1]([O:2][H:4])[H:3]
1,[C:1]([C:2]([F:3])([H:7])[H:8])([H:4])([H:5])[...


Contains 344164 rows


In [13]:
molecules = swifter_apply(
    unique_solutes["atom_mapped_smiles"],
    lambda x: molecule_from_smiles(x, remove_atom_mapping=True),
    desc="Generating mols",
)

unique_solutes["canonical_smiles"] = swifter_apply(
    molecules,
    Chem.MolToSmiles,
    desc="Generating canonical smiles",
)

unique_solutes["num_radical_electrons"] = swifter_apply(
    molecules,
    Descriptors.NumRadicalElectrons,
    desc="Detecting radical electrons",
)

head(unique_solutes)

Generating mols:   0%|          | 0/344164 [00:00<?, ?it/s]

Generating canonical smiles:   0%|          | 0/344164 [00:00<?, ?it/s]

Detecting radical electrons:   0%|          | 0/344164 [00:00<?, ?it/s]

,atom_mapped_smiles,canonical_smiles,num_radical_electrons
0,[O:1]([O:2][H:4])[H:3],OO,0
1,[C:1]([C:2]([F:3])([H:7])[H:8])([H:4])([H:5])[...,CCF,0


Contains 344164 rows


In [14]:
solute_to_canonical_smiles_mapping = unique_solutes.set_index("atom_mapped_smiles")[
    "canonical_smiles"
].to_dict()

solute_to_num_radical_electrons_mapping = unique_solutes.set_index(
    "atom_mapped_smiles"
)["num_radical_electrons"].to_dict()

In [15]:
data["smiles"] = data["solute_smiles"].map(solute_to_canonical_smiles_mapping.get)
data["num_radical_electrons"] = data["solute_smiles"].map(
    solute_to_num_radical_electrons_mapping.get
)
head(data)

,solvent_smiles,solute_name,solute_smiles,Gsolv (kcal/mol),Hsolv (kcal/mol),project,hash,solvent_name,smiles,num_radical_electrons
0,O,id0,[O:1]([O:2][H:4])[H:3],-7.178090,-11.557472,aug11b,InChI=1/H2O2/c1-2/h1-2HInChI=1S/H2O/h1H2,h2o,OO,0
1,CCCCCCCCO,id0,[O:1]([O:2][H:4])[H:3],-5.797229,-14.171550,aug11b,InChI=1/H2O2/c1-2/h1-2HInChI=1S/C8H18O/c1-2-3-...,1-octanol,OO,0


Contains 101513860 rows


In [16]:
ouput_columns = ["smiles", "Gsolv (kcal/mol)", "Hsolv (kcal/mol)"]

In [17]:
grouped_solvents = data.groupby(["solvent_name", "num_radical_electrons"])

In [18]:
dirnames = {
    0: "closed_shell_species",
    1: "open_shell_species",
}
for dirname in dirnames.values():
    (DATA_DIR / dirname).mkdir(parents=True, exist_ok=True)

for (name, num_radical_electrons), group in tqdm(
    grouped_solvents, total=2 * len(unique_solvents)
):
    subdir = DATA_DIR / dirnames[num_radical_electrons] / first_letter(name)
    subdir.mkdir(parents=True, exist_ok=True)
    filename = f"{name}.csv"
    group[ouput_columns].to_csv(subdir / filename, index=False, float_format="%.10g")

  0%|          | 0/590 [00:00<?, ?it/s]